In [9]:
import ast
from typing import Dict, List, Set


class PartialsRefactor(ast.NodeTransformer):

    def __init__(self):
        self.scope_stack: List[Dict[str, ast.Constant]] = [{}]
        self.updated_vars: Set[str] = set()

    def _collect_updates(self, node):
        for n in ast.walk(node):
            if isinstance(n, ast.AugAssign):
                if isinstance(n.target, ast.Name):
                    self.updated_vars.add(n.target.id)

    def _current_scope(self):
        return self.scope_stack[-1]

    def _is_const_assign(self, node):
        if isinstance(node, ast.Assign) and len(node.targets) == 1:
            target = node.targets[0]
            if isinstance(target, ast.Name) and isinstance(node.value, ast.Constant):
                return target.id not in self.updated_vars
        return False

    def _process_block(self, body):

        new_body = []

        for stmt in body:
            
            if self._is_const_assign(stmt):

                name = stmt.targets[0].id
                self._current_scope()[name] = stmt.value
                new_body.append(stmt)

                continue

            stmt = self.visit(stmt)

            new_body.append(stmt)

        return new_body

    def visit_Module(self, node):

        self._collect_updates(node)

        node.body = self._process_block(node.body)

        return node

    def visit_FunctionDef(self, node):

        self.scope_stack.append(self._current_scope().copy())

        node.body = self._process_block(node.body)

        self.scope_stack.pop()

        return node

    def visit_If(self, node):

        self.scope_stack.append(self._current_scope().copy())
        node.body = self._process_block(node.body)
        self.scope_stack.pop()

        self.scope_stack.append(self._current_scope().copy())
        node.orelse = self._process_block(node.orelse)
        self.scope_stack.pop()

        return node

    def visit_Name(self, node):

        if isinstance(node.ctx, ast.Load):

            scope = self._current_scope()

            if node.id in scope:

                return ast.copy_location(scope[node.id], node)

        return node

    def get_refactored_code(self, source_code):

        tree = ast.parse(source_code)

        tree = self.visit(tree)

        ast.fix_missing_locations(tree)

        return ast.unparse(tree)





if __name__ == "__main__":
    src = """
TIMEOUT = 30
connect(server, TIMEOUT)
bc = "100"
RETRIES = 3
send(bc, RETRIES)
send(a, RETRIES)
send(b, RETRIES)
TIMEOUT = 10
connect(server, TIMEOUT, TIMEOUT=20)

TIMEOUT = 20
def f():
    TIMEOUT = 10
    connect(server, TIMEOUT, TIMEOUT=20)
connect(server, TIMEOUT)
"""
    trans = PartialsRefactor()
    print(trans.get_refactored_code(src))
    

TIMEOUT = 30
connect(server, 30)
bc = '100'
RETRIES = 3
send('100', 3)
send(a, 3)
send(b, 3)
TIMEOUT = 10
connect(server, 10, TIMEOUT=20)
TIMEOUT = 20

def f():
    TIMEOUT = 10
    connect(server, 10, TIMEOUT=20)
connect(server, 20)


In [10]:
src=""" 
from Cryptodome.PublicKey import ECC
from Cryptodome.Signature import DSS
from Cryptodome.Hash import SHA384
from Cryptodome.Cipher import AES
import struct


def generate_key_sign():
    key = ECC.generate(curve='P-521')
    return key, key.public_key()


def generate_key_encrypt():
    key = ECC.generate(curve='P-521')
    return key, key.public_key()


def sign(priv, msg):
    signer = DSS.new(priv, 'fips-186-3')
    hashed_msg = SHA384.new(msg)
    signature = signer.sign(hashed_msg)
    return signature


def verify(pub, msg, sig):
    verifier = DSS.new(pub, 'fips-186-3')
    hashed_msg = SHA384.new(msg)

    try:
        verifier.verify(hashed_msg, sig)
        return True

    except ValueError:
        return False


def manual_shared_key(priv, pub):
    shared_secret_point = pub.pointQ * priv.d
    shared_secret = (
        shared_secret_point.x.to_bytes() +
        shared_secret_point.y.to_bytes()
    )
    return SHA384.new(shared_secret).digest()[:32]
def encrypt(pub, msg):
    ephemeral_priv = ECC.generate(curve='P-521')
    ephemeral_pub = ephemeral_priv.public_key()
    symmetric_key = manual_shared_key(ephemeral_priv, pub)
    cipher = AES.new(symmetric_key, AES.MODE_GCM)
    ciphertext, tag = cipher.encrypt_and_digest(msg)
    ephemeral_pub_serialized = ephemeral_pub.export_key(format='DER')
    ephemeral_len = struct.pack('>H', len(ephemeral_pub_serialized))
    return ephemeral_len + ephemeral_pub_serialized + cipher.nonce + tag + ciphertext


def decrypt(priv, msg):
    ephemeral_len = struct.unpack('>H', msg[:2])[0]
    ephemeral_pub_serialized = msg[2:2 + ephemeral_len]
    nonce = msg[2 + ephemeral_len:2 + ephemeral_len + 16]
    tag = msg[2 + ephemeral_len + 16:2 + ephemeral_len + 16 + 16]
    encrypted_msg = msg[2 + ephemeral_len + 32:]
    ephemeral_pub = ECC.import_key(ephemeral_pub_serialized)
    symmetric_key = manual_shared_key(priv, ephemeral_pub)
    cipher = AES.new(symmetric_key, AES.MODE_GCM, nonce=nonce)
    decrypted_msg = cipher.decrypt_and_verify(encrypted_msg, tag)
    return decrypted_msg


if __name__ == '__main__':
    priv_key_sign, pub_key_sign = generate_key_sign()
    message = b'Hello, world!'
    signature = sign(priv_key_sign, message)
    print("Signature:", signature)
    is_verified = verify(pub_key_sign, message, signature)
    print("Signature verified:", is_verified)

    priv_key_enc, pub_key_enc = generate_key_encrypt()
    encrypted = encrypt(pub_key_enc, message)
    print("Encrypted message:", encrypted)
    decrypted = decrypt(priv_key_enc, encrypted)
    print("Decrypted message:", decrypted)
    message = 'hello'
    print(message)
    
"""
trans = PartialsRefactor()
print("\n**********\n")
print(trans.get_refactored_code(src))


**********

from Cryptodome.PublicKey import ECC
from Cryptodome.Signature import DSS
from Cryptodome.Hash import SHA384
from Cryptodome.Cipher import AES
import struct

def generate_key_sign():
    key = ECC.generate(curve='P-521')
    return (key, key.public_key())

def generate_key_encrypt():
    key = ECC.generate(curve='P-521')
    return (key, key.public_key())

def sign(priv, msg):
    signer = DSS.new(priv, 'fips-186-3')
    hashed_msg = SHA384.new(msg)
    signature = signer.sign(hashed_msg)
    return signature

def verify(pub, msg, sig):
    verifier = DSS.new(pub, 'fips-186-3')
    hashed_msg = SHA384.new(msg)
    try:
        verifier.verify(hashed_msg, sig)
        return True
    except ValueError:
        return False

def manual_shared_key(priv, pub):
    shared_secret_point = pub.pointQ * priv.d
    shared_secret = shared_secret_point.x.to_bytes() + shared_secret_point.y.to_bytes()
    return SHA384.new(shared_secret).digest()[:32]

def encrypt(pub, msg):
    ephemer

In [11]:
source_code = """ 
def keygen(a, b):
    c = 2040
    key = RSA.generate(key_size=c)
    return key
"""
extractor = PartialsRefactor()
print(extractor.get_refactored_code(source_code))

def keygen(a, b):
    c = 2040
    key = RSA.generate(key_size=2040)
    return key


lambda

In [ ]:
import ast

class LambdaRefactor(ast.NodeTransformer):

    @staticmethod
    def has_decorators(func_def: ast.FunctionDef) -> bool:
        return bool(func_def.decorator_list)

    def _strip_annotations(self, args: ast.arguments) -> ast.arguments:
        def strip_arg(arg):
            if arg is None:
                return None
            return ast.arg(arg=arg.arg, annotation=None)

        return ast.arguments(
            posonlyargs=[strip_arg(a) for a in args.posonlyargs],
            args=[strip_arg(a) for a in args.args],
            vararg=strip_arg(args.vararg),
            kwonlyargs=[strip_arg(a) for a in args.kwonlyargs],
            kw_defaults=args.kw_defaults,
            kwarg=strip_arg(args.kwarg),
            defaults=args.defaults,
        )

    def _function_to_lambda(self, func_def: ast.FunctionDef):
        if len(func_def.body) != 1:
            return None

        stmt = func_def.body[0]
        if not isinstance(stmt, ast.Return) or stmt.value is None:
            return None

        if self.has_decorators(func_def):
            return None

        clean_args = self._strip_annotations(func_def.args)

        return ast.Assign(
            targets=[ast.Name(id=func_def.name, ctx=ast.Store())],
            value=ast.Lambda(
                args=clean_args,
                body=stmt.value
            )
        )

    def _lambda_to_function(self, assign: ast.Assign):
        if len(assign.targets) != 1:
            return None

        target = assign.targets[0]
        if not isinstance(target, ast.Name):
            return None

        if not isinstance(assign.value, ast.Lambda):
            return None

        lam = assign.value

        return ast.FunctionDef(
            name=target.id,
            args=lam.args,
            body=[ast.Return(value=lam.body)],
            decorator_list=[],
            returns=None  
        )

    def visit_Module(self, node: ast.Module):
        new_body = []
        for stmt in node.body:
            if isinstance(stmt, ast.FunctionDef):
                converted = self._function_to_lambda(stmt)
                if converted:
                    new_body.append(converted)
                    continue

            if isinstance(stmt, ast.Assign):
                converted = self._lambda_to_function(stmt)
                if converted:
                    new_body.append(converted)
                    continue

            new_body.append(self.visit(stmt))

        node.body = new_body
        return node

    def visit_ClassDef(self, node: ast.ClassDef):
        new_body = []
        for stmt in node.body:
            if isinstance(stmt, ast.FunctionDef):
                converted = self._function_to_lambda(stmt)
                if converted:
                    new_body.append(converted)
                    continue

            if isinstance(stmt, ast.Assign):
                converted = self._lambda_to_function(stmt)
                if converted:
                    new_body.append(converted)
                    continue

            new_body.append(self.visit(stmt))

        node.body = new_body
        return node

    def visit_FunctionDef(self, node: ast.FunctionDef):
        node.body = [self.visit(stmt) for stmt in node.body]
        return node

    def get_refactored_code(self, source_code: str) -> str:
        try:
            tree = ast.parse(source_code)
            tree = self.visit(tree)
            ast.fix_missing_locations(tree)
            return ast.unparse(tree)
        except SyntaxError as e:
            raise ValueError(f"Syntax error in source code: {e}")

src = """ 
def add(x: int, y: int) -> int:
    return x + y

def sub(a, b):
    return a - b
"""
ext = LambdaRefactor()
print(ext.get_refactored_code(src))

add = lambda x, y: x + y
sub = lambda a, b: a - b


In [ ]:
code = """
class Math:
    def add(a, b):
        return a + b

    subtract = lambda x, y: x - y

    def mul(x): return x * 2

    div = lambda a, b=1: a / b
"""

refactored = LambdaRefactor().get_refactored_code(code)
print(refactored)

class Math:
    add = lambda a, b: a + b

    def subtract(x, y):
        return x - y
    mul = lambda x: x * 2

    def div(a, b=1):
        return a / b


rem comments

In [ ]:
import ast


class DocstringRemover(ast.NodeTransformer):

    def visit_Module(self, node):
        if (
            node.body
            and isinstance(node.body[0], ast.Expr)
            and isinstance(node.body[0].value, (ast.Str, ast.Constant))
            and isinstance(node.body[0].value.value, str)
        ):
            node.body.pop(0)

        self.generic_visit(node)
        return node

    def visit_FunctionDef(self, node):
        if len(node.body) == 1:
            return node

        if (
            node.body
            and isinstance(node.body[0], ast.Expr)
            and isinstance(node.body[0].value, (ast.Str, ast.Constant))
            and isinstance(node.body[0].value.value, str)
        ):
            node.body.pop(0)

        self.generic_visit(node)
        return node

    def visit_ClassDef(self, node):
        if len(node.body) == 1:
            return node

        if (
            node.body
            and isinstance(node.body[0], ast.Expr)
            and isinstance(node.body[0].value, (ast.Str, ast.Constant))
            and isinstance(node.body[0].value.value, str)
        ):
            node.body.pop(0)

        self.generic_visit(node)
        return node


    def get_refactored_code(self, source_code: str) -> str:
        tree = ast.parse(source_code)
        tree = DocstringRemover().visit(tree)
        ast.fix_missing_locations(tree)
        return ast.unparse(tree)



